In [ ]:
from typing import List, TypedDict

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
docs = (
    PyPDFLoader("Hands On Machine Learning with Scikit Learn and TensorFlow.pdf").load() +
    PyPDFLoader("Python_Datascience.pdf").load()
)

In [ ]:
len(docs)

In [ ]:
docs[0].metadata

In [ ]:
docs[0].page_content

In [ ]:
# 2) Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
len(chunks)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
# 4) LLM + prompt
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
def rewrite_query(user_query):
    prompt = f"""
    Rewrite the following query to make it more detailed and specific for document retrieval.
    donot change the meaning of actual query

    Query: {user_query}

    Rewritten Query:
    """

    response = llm.invoke(prompt)
    return response.text()

In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
class State(TypedDict):
    question: str
    rewritten_query: str
    docs: List[Document]
    answer: str

In [ ]:
def rerank_documents(query, docs):
    pairs = [(query, doc.page_content) for doc in docs]

    scores = reranker.predict(pairs)

    # attach scores
    doc_scores = list(zip(docs, scores))

    # sort by score (descending)
    doc_scores.sort(key=lambda x: x[1], reverse=True)

    # return top 5 docs
    return doc_scores[:5]

In [ ]:
def retrieve(state):
    original_q = state["question"]
    rewritten_q = rewrite_query(original_q)
    docs = vector_store.similarity_search(rewritten_q, k=15)
    # Step 2: rerank
    reranked = rerank_documents(original_q, docs)
    print("\n RERANKING RESULTS:\n")

    final_docs = []

    for i, (doc, score) in enumerate(reranked):
        print(f"Rank {i+1} | Score: {score:.4f}")
        print(doc.page_content[:200])
        print("-" * 50)

        final_docs.append(doc)

    return {
        "docs": final_docs,
        "rewritten_query": rewritten_q
    }

In [ ]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}


In [ ]:
from langgraph.graph import StateGraph,START,END

In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
# 5) Run
res = app.invoke({"question": "WHat is difference between numpy and pandas.", "docs": [], "answer": "","rewritten_query":{}})

In [ ]:
res["question"]

In [ ]:
res["rewritten_query"]

In [ ]:
res["answer"]

In [ ]:
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)